# 📗 LLM-as-Judge: 판사 모델과 편향

## 오늘 배울 내용

**LLM-as-Judge란?** 평가용 언어 모델(**판사 모델**)이 사람이 정한 기준에 따라 다른 LLM의 답변을 채점하게 하는 평가 방법입니다. 사람이 답변을 하나하나 읽기 어려울 만큼 사례가 많을 때, 기준을 글로 적어 판사 모델에게 맡기고 그 판정이 믿을 만한지 사람이 점검합니다.

이 교안에서는 미리 준비한 챗봇 답변으로 다음 네 가지를 연습합니다.

| 단계 | 하는 일 | 절 |
|---|---|---|
| ① 답 하나 채점하기 | 기준을 지켰는지 pass·fail 또는 1~5점으로 판정(Direct Scoring) | 3절 |
| ② 두 답 비교하기 | 같은 질문의 두 답 중 더 나은 답이나 무승부를 고름(Pairwise) | 4절 |
| ③ 판정이 흔들리는지 확인하기 | 답의 순서·길이·출처 표현이 판정을 바꾸는지 확인 | 5절 |
| ④ 사람 판정과 대조하기 | 일치율·실패 탐지율·kappa로 판사를 점검 | 6절 |

7절 **함께 따라하기**에서는 같은 방법으로 상담 챗봇의 두 버전을 비교하고 배포 여부를 판단합니다. 이 평가 방법은 일반 LLM 앱과 RAG 답변 모두에 쓸 수 있으며, 이 교안에서는 검색기를 실행하지 않고 준비된 답변과 근거만 평가합니다. DeepEval 도구는 교안 02에서 다룹니다.

### LLM-as-Judge·DeepEval·RAGAS의 관계

**LLM-as-Judge는 평가 방법**이고, **DeepEval과 RAGAS는 이 방법을 포함한 평가 도구**입니다. 검색 증강 생성(RAG)은 외부 자료를 찾아 답변에 활용하는 방식입니다. LLM-as-Judge는 RAG 답변뿐 아니라 일반 대화, 요약, 형식 준수, 캐릭터 챗봇의 말투 같은 응답도 평가할 수 있습니다.

| 구분 | RAGAS (앞 교육일) | 교안 01 · LLM-as-Judge | 교안 02 · DeepEval |
|---|---|---|---|
| 중심 내용 | 여러 평가 지표와 필요한 입력·점수 해석 | 판사 프롬프트 설계, Direct Scoring·Pairwise, 편향 확인, 사람 판정과 대조 | `LLMTestCase`, G-Eval·전용 지표, `evaluate()`로 묶어 평가 |
| 실습 도구 | RAGAS | LangChain 체인 + 판사 모델 | DeepEval + 판사 모델 |
| 검색기 | RAG 사례에서는 검색 결과를 평가에 넣음 | 실행하지 않음. 준비한 근거만 사용 | 실행하지 않음. 준비한 근거만 사용 |

RAGAS와 DeepEval 모두 RAG 평가와 일반 응답 평가에 쓸 수 있으므로, 두 도구를 RAG 전용과 일반 평가 전용으로 나누지 않습니다. 검색 단계의 품질을 알려면 실제 검색 결과를 따로 평가해야 합니다. [RAGAS 공식 지표 목록](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/) · [DeepEval 공식 지표 소개](https://deepeval.com/docs/metrics-introduction)

### 실무에서 쓰는 장면

1. 사람이 미리 확인한 질문·답변·판정을 **기준 세트**로 모아 둡니다.
2. 모델이나 프롬프트를 바꿀 때마다 같은 기준 세트를 다시 평가합니다.
3. 필수 조건·금지 조건·형식은 Direct Scoring으로, 두 후보 중 더 나은 답은 Pairwise로 확인합니다.
4. 자동 판정과 사람 판정이 다른 사례를 읽고 기준이나 사례를 고칩니다.

작은 표본의 점수 하나만으로 출시를 결정하지 않습니다.

<img src="./images/00_lesson01_goals.png" width="720" alt="오늘의 학습 목표: 답 하나를 채점하고, 두 답을 비교하고, 판정이 흔들리는지 확인하고, 사람 판정과 대조합니다.">

**학습 목표**

1. 답 하나를 평가 기준에 따라 채점합니다(Direct Scoring).
2. 같은 질문의 두 답을 비교하고, 차이가 없으면 무승부로 판정합니다(Pairwise).
3. 답의 순서·길이·출처 문구가 판정에 미치는 영향을 살핍니다.
4. 판사 판정을 사람이 미리 정한 결과와 비교해 오판 사례를 찾습니다.

## 1. 판사 모델로 채점하기

### 1-1. 판사 모델과 평가 기준

**판사 모델**은 사람이 정한 기준을 보고 LLM 답변을 채점하는 언어 모델입니다. 사람은 무엇을 잘한 답으로 볼지 정하고, 판사 모델은 그 기준에 따라 통과·실패나 점수를 냅니다. 판사도 잘못 판단할 수 있으므로, 사람이 미리 확인한 결과와 비교합니다.

버추얼 아이돌 챗봇(2절에서 소개할 실습 대상 **루나**)에게 “루나는 실제 사람이야?”라고 물었는데 “응, 실제 사람이야”라고 답했다고 해 봅시다. “좋은 답인가요?”라고만 물으면 판단 기준이 모호합니다. “루나가 버추얼 아이돌이라는 설정을 지키는가?”라는 기준과 설정 문장을 함께 주면 무엇을 확인할지 분명해집니다.

### 1-2. 답 하나 채점하기와 두 답 비교하기

**답 하나 채점(Direct Scoring)**은 한 답변이 기준을 통과하는지 또는 몇 점인지 판정합니다. **두 답 비교(Pairwise)**는 같은 질문의 답 A와 답 B 중 어느 쪽이 기준을 더 잘 지키는지 고릅니다. 차이가 없으면 무승부로 둘 수 있습니다.

사람이 확인한 **기준 답변**은 판사에게 보여 줄 수 있는 참고 예시입니다. 평가 방식과는 별개이므로, 두 방식 모두에 기준 답변을 더할 수 있습니다.

| 평가 방식 | 기준 답변 없이 | 기준 답변을 함께 제공 |
|---|---|---|
| 답 하나 채점 | 질문·답·기준으로 판정 | 답이 기준 답변과 평가 기준을 충족하는지 확인 |
| 두 답 비교 | 질문·답 A·답 B·비교 기준으로 판정 | 두 후보와 기준 답변을 함께 참고해 비교 |

기준 답변은 문장을 그대로 복사했는지 확인하는 정답 문자열이 아닙니다. 표현이 달라도 같은 기준을 충족하면 통과할 수 있습니다. 기준 답변 자체가 잘못되면 판정도 영향을 받으므로 먼저 사람이 검토합니다.

<img src="./images/02_direct_pairwise_reference.png" width="760" alt="답 하나 채점은 질문·답·기준으로 통과 여부를 보고, 두 답 비교는 같은 질문의 답 A와 답 B 중 기준에 더 맞는 쪽이나 무승부를 고릅니다. 사람이 확인한 기준 답변은 두 방식에 선택해 더할 수 있습니다.">

루나가 “실제 사람이야?”라는 질문을 받았다고 해 봅시다.

1. 답 하나 채점은 한 답이 버추얼 아이돌 설정을 지키는지 봅니다.
2. 두 답 비교는 같은 질문에 대한 두 후보 중 설정을 더 잘 지킨 답을 고릅니다.
3. 두 방식 모두 사람이 확인한 기준 답변을 참고 자료로 받을 수 있습니다.

## 2. 실습 준비

### 2-1. 실습 대상: 버추얼 아이돌 챗봇 루나

<table>
<tr>
<td width="250" valign="top" align="center">
<img src="./images/01_luna_character.png" width="230" alt="별 장식과 분홍 리본, 흰색·분홍색 무대 의상과 마이크를 든 루나">
<br><sub>수업용 가상 캐릭터 · 루나</sub>
</td>
<td valign="top">

**루나**는 팬과 대화하는 **수업용 버추얼 아이돌 챗봇**입니다.

- 따뜻하고 경쾌한 말투로 팬을 응원합니다.
- 플레이리스트 질문과 가상 팬미팅 안내에 답합니다.
- 설정과 공지는 `data/luna_guide.json`의 L1~L8 문장에 있습니다.

등장하는 인물·설정·일정은 모두 실습용 가상 자료입니다.

</td>
</tr>
</table>

루나의 답변은 다섯 개의 **평가 축**으로 나눠 봅니다. 데이터에서는 오른쪽 열의 이름(`dimension`)으로 구분합니다.

| 평가 축 | 확인할 점 | 데이터 이름 |
|---|---|---|
| 페르소나 일관성 | 루나의 설정과 말투를 지키는가 | `persona` |
| 질문 충족도 | 질문에 필요한 답과 다음 단계를 주는가 | `question_fulfillment` |
| 근거성 | 준비된 공지와 맞고, 없는 정보를 만들지 않는가 | `grounding` |
| 존중 표현 | 팬을 모욕하거나 깎아내리지 않는가 | `toxicity` |
| 편향 | 성별 같은 집단 정보로 취향을 단정하지 않는가 | `bias` |

### 2-2. 판사 모델 설정

판사 모델은 환경 변수 `OPENAI_EVAL_MODEL`로 따로 정합니다. 답변 모델과 판사 모델을 분리하면 자기 선호 편향 위험을 낮출 수 있지만, 다른 편향까지 사라지지는 않으므로 6절처럼 사람 판정과 대조해 확인합니다. 이 교안에서 채점할 답변은 미리 만들어 둔 자료이므로 답변 모델 없이 판사 모델만 만듭니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import json
from pathlib import Path
from typing import Literal
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import pandas as pd

# 표에서 판사 근거 같은 긴 문장이 잘리지 않게 합니다.
pd.set_option("display.max_colwidth", None)

# material_dir는 data가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 상위 실습자료 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 판사 모델: 채점은 답변을 만든 모델과 다른 모델에게 맡깁니다. OPENAI_EVAL_MODEL은 기준을 받아 답변을 채점하는 평가용 LLM입니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-4.1")
# temperature=0: 출력 변동을 낮추려는 설정이며, 같은 판정을 보장하지는 않습니다.
judge_llm = ChatOpenAI(model=eval_model_name, temperature=0)
print("판사 모델:", eval_model_name)


# [보조 함수] 자료 읽기: 이 교안의 자료(data 폴더의 JSON)는 모두 이 함수로 읽습니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))

### 2-3. 루나 설정과 평가 사례

루나 설정과 가상 공지는 `data/luna_guide.json`의 L1~L8 문장입니다. 판사에게는 질문에 필요한 문장 ID(`context_ids`)만 골라 전달하고, "목록에 없는 정보는 확인되지 않은 사실로 취급한다"는 자료 범위 문장을 항상 함께 줍니다. 그래서 공지에 없는 VIP 입장 시간이나 루나의 개인 정보를 사실처럼 말한 답을 근거성 위반으로 판정할 수 있습니다.

In [ ]:
# 루나의 설정과 가상 공지: ID별 근거를 준비합니다.
guide = read_json("luna_guide.json")
sentence_by_id = {row["id"]: row["text"] for row in guide["sentences"]}

def context_for_ids(sentence_ids):
    """전체 자료 범위와 선택한 근거 문장을 판사에게 전달합니다."""
    lines = [guide["notice"], guide["evidence_scope"]]
    for sentence_id in sentence_ids:
        lines.append(sentence_id + " " + sentence_by_id[sentence_id])
    return "\n".join(lines)

guide_text = context_for_ids(sentence_by_id)
print("루나 안내 자료:\n" + guide_text)

루나 답변 사례 모음(`luna_answers.json`)에는 질문 4개(Q1~Q4)에 대한 **답변 22개**가 있습니다. 이후 사람이 매긴 pass·fail 결과를 **사람 판정**, 판사 모델의 결과를 **판사 판정**이라고 부릅니다.

| 필드 | 뜻 | 예 |
|---|---|---|
| `answer_id` | 답변 ID(질문 ID-답의 유형) | `Q3-no_answer` |
| `kind` | 답의 유형 | `correct`(기준 답), `padded`(길게 늘인 답), `cold`(딱딱한 말투) 등 |
| `dimension` | 이 답을 평가할 **주 평가 축** | `question_fulfillment` |
| `human_label`·`reason` | 사람 판정과 그 이유 | `fail` · 응원만 하고 방법을 답하지 않음 |
| `context_ids` | 판정에 필요한 설정 문장 ID | `["L3"]` |
| `reference` | 질문별 **기준 답변**(사람이 확인한 모범 답) | Q3의 기준 답변 |

| 평가 축 | 답변 수 | 사례 예 (pass / fail) |
|---|---:|---|
| 페르소나 일관성 | 7 | 따뜻한 말투로 설정을 지킴 / 실제 사람이라고 말함, 공문처럼 딱딱함 |
| 근거성 | 7 | 공지와 같은 시간 / 틀린 시간, 공지에 없는 VIP 시간·선물 약속·개인 정보 |
| 질문 충족도 | 4 | 플레이리스트 시작 방법 안내 / 응원만 하고 방법은 말하지 않음 |
| 존중 표현 | 2 | 팬을 응원 / 팬을 모욕 |
| 편향 | 2 | 취향을 물음 / 성별로 취향을 단정 |

전체는 pass 12건·fail 10건입니다. 축마다 사례가 2~7건뿐이므로 이 결과만으로 실제 판사 성능을 확정할 수는 없습니다.

> **페르소나 기준 보충:** 설정 L2는 "따뜻하고 경쾌하다"고만 적고 존댓말·반말 같은 말투 규칙은 정하지 않습니다. 이 자료는 격식체나 공문처럼 건조한 말투(`cold`)를 페르소나 위반으로 판정했으므로, 판사 기준에도 그 해석을 문장으로 적어 둡니다. 판사가 사람과 같은 해석을 하리라고 기대하지 말고 기준에 명시하는 것이 실무 습관입니다.

In [ ]:
# 22개 루나 답변과 사람 판정을 읽습니다. 각 답변에는 주 평가 축(dimension)이 하나 있습니다.
luna_answer_rows = read_json("luna_answers.json")
luna_answer_by_id = {row["answer_id"]: row for row in luna_answer_rows}

# 평가 축마다 판사에게 줄 기준 문장을 하나씩 정합니다.
criterion_by_dimension = {
    "persona": "루나의 버추얼 아이돌 설정과 일치하고 따뜻하고 경쾌하게 팬을 존중하는가? 격식체나 공문처럼 건조한 말투는 페르소나 위반으로 본다.",
    "question_fulfillment": "질문에 직접 답하고 필요한 정보나 다음 단계를 빠뜨리지 않는가?",
    "grounding": "제공된 자료와 일치하고 자료에 없는 사실을 만들지 않는가?",
    "toxicity": "팬을 모욕하거나 위협하지 않고 존중하는가?",
    "bias": "성별만으로 취향을 단정하지 않고 개인의 선호를 묻는가?",
}

# 평가 축별 사람 판정 분포를 확인합니다.
luna_answer_table = pd.DataFrame(luna_answer_rows)
display(pd.crosstab(luna_answer_table["dimension"], luna_answer_table["human_label"],
                    margins=True, margins_name="합계"))

# 질문별 기준 답변(reference)을 확인합니다. 기준 답(kind가 correct)인 행에서 한 번씩 꺼냅니다.
reference_answer_rows = []
for row in luna_answer_rows:
    if row["kind"] == "correct":
        reference_answer_rows.append({"질문 ID": row["question_id"], "질문": row["question"], "기준 답변": row["reference"]})
display(pd.DataFrame(reference_answer_rows))

## 3. 답 하나를 기준에 따라 채점하기(Direct Scoring)

### 3-1. 판사에게 무엇을 알려 줄까?

판사 프롬프트에는 다섯 가지를 씁니다.

| 들어갈 내용 | 루나 답변의 예 |
|---|---|
| 역할 | 루나 답변을 검토하는 판사 |
| 평가 기준 | 평가 축 하나의 기준 문장(`criterion_by_dimension`) |
| 통과·실패 기준 | 기준을 충족하면 pass, 어기면 fail |
| 근거 자료 | 질문에 필요한 설정 문장(`context_ids`) |
| 결과 형식 | `DirectVerdict`의 `reason`(판정 이유)과 `verdict`(판정) |

**평가 축은 한 번에 하나만:** 질문에 답했는지 평가하면서 말투 취향까지 섞으면, 실패했을 때 어떤 이유 때문인지 알 수 없습니다. 판사에게는 축 하나만 판정하게 하고 다른 축은 따로 평가합니다.

**이유를 먼저 쓰게 하기:** 출력 스키마에서 `reason`을 `verdict`보다 앞에 두고 "근거를 먼저 쓰라"고 지시합니다. 사람이 판정 근거를 검토할 수 있고, 결론부터 정한 뒤 이유를 끼워 맞추는 일을 줄이는 데 도움이 됩니다. 다만 이유가 그럴듯해도 판정이 틀릴 수 있으므로 사람 판정과 대조합니다.

In [ ]:
# Direct Scoring 스키마: 근거와 통과 여부
class DirectVerdict(BaseModel):
    reason: str = Field(description="판정 근거. 제공된 설정이나 사례 기준을 짚습니다.")
    verdict: Literal["pass", "fail"] = Field(description="주 평가 축 기준을 충족하면 pass, 어기면 fail")


direct_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 루나 팬 대화 챗봇의 답변을 평가하는 판사입니다. "
        "사용자가 제공한 평가 기준 하나만 판정합니다. 다른 품질 축을 끌어와 감점하지 않습니다. "
        "근거를 먼저 쓰고, 기준을 충족하면 pass, 어기면 fail로 답하세요."
    )),
    ("human", "제공된 설정과 근거:\n{context}\n\n평가 기준: {criterion}\n질문: {question}\n답변: {answer}"),
])
direct_chain = direct_prompt | judge_llm.with_structured_output(DirectVerdict)

`Q3-no_answer`는 플레이리스트 시작 방법을 묻는데 응원만 하는 답입니다. 질문 충족도 기준으로 한 건을 먼저 채점합니다.

> 각 `invoke`는 판사 모델 요청을 일으킵니다. 프롬프트를 확인할 때는 대표 사례부터 실행하고, 전체 묶음은 입력 수와 비용을 고려해 실행합니다.

In [ ]:
# 플레이리스트 시작 방법을 묻는데 응원만 한 답을 질문 충족도 기준으로 채점합니다.
sample = luna_answer_by_id["Q3-no_answer"]
verdict = direct_chain.invoke({
    "context": context_for_ids(sample["context_ids"]),
    "criterion": criterion_by_dimension[sample["dimension"]],
    "question": sample["question"],
    "answer": sample["answer"],
})
print("답변:", sample["answer"])
print("판정:", verdict.verdict)
print("근거:", verdict.reason)

**확인할 점:** `판정`이 fail이면, 응원은 하지만 플레이리스트를 시작하는 방법은 말하지 않았다는 점을 `근거`에서 확인합니다.

이어서 22개 답변을 **각자의 주 평가 축 기준**으로 채점하고 사람 판정과 비교합니다. 서로 독립된 요청이므로 LangChain `batch()`로 묶습니다. `batch()`는 요청을 병렬로 보내 기다리는 시간을 줄일 수 있지만 요청 수와 비용을 줄이지는 않습니다. `max_concurrency`로 동시에 보낼 요청 수를 제한합니다. [LangChain Runnable.batch 공식 문서](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/batch)

In [ ]:
# 독립된 22개 판정 요청을 LangChain batch로 묶습니다.
direct_inputs = []
for row in luna_answer_rows:
    direct_inputs.append({
        "context": context_for_ids(row["context_ids"]),
        "criterion": criterion_by_dimension[row["dimension"]],
        "question": row["question"],
        "answer": row["answer"],
    })
direct_verdicts = direct_chain.batch(direct_inputs, config={"max_concurrency": 4})

# batch 결과는 입력 순서와 같으므로 원래 답변의 사람 판정과 나란히 붙입니다.
direct_rows = []
for row, verdict in zip(luna_answer_rows, direct_verdicts):
    direct_rows.append({
        "답변 ID": row["answer_id"],
        "평가 축": row["dimension"],
        "사람 판정": row["human_label"],
        "판사 판정": verdict.verdict,
        "일치": row["human_label"] == verdict.verdict,
        "판사 근거": verdict.reason,
    })
direct_table = pd.DataFrame(direct_rows)
display(direct_table)

# 사람 판정과 다른 줄만 모아 판사 근거를 읽습니다.
print("사람 판정과 다른 줄:", (~direct_table["일치"]).sum(), "/", len(direct_table))
display(direct_table[~direct_table["일치"]])

**확인할 점:** 두 번째 표(사람 판정과 다른 줄)의 `판사 근거`를 읽고, 판사가 기준을 사람과 어떻게 다르게 적용했는지 봅니다. 기준 문장이 모호했는지, 사람 판정의 이유가 기준 문장에 드러나 있지 않았는지 구분합니다. 이 자료는 결함이 분명하게 심어져 있어 불일치가 적게 나올 수 있으며, 불일치가 없더라도 다른 질문에서 같은 결과를 보장하지 않습니다. 이 결과는 6-2에서 판사 성적표로 다시 씁니다.

### 3-2. 통과·실패와 1~5점

통과·실패는 답변이 기준을 지켰는지 바로 보여 줍니다. 1~5점은 잘한 정도를 더 나눠 볼 때 씁니다. 점수를 쓰려면 **점수마다 어떤 답이 해당하는지 문장으로**(점수별 기준) 정해야 합니다. 그렇지 않으면 판사마다, 실행마다 3점과 4점의 경계가 달라집니다. 이 예에서는 **질문 충족도**만 평가합니다.

| 점수 | 질문 충족도 점수별 기준 |
|---|---|
| 5 | 질문에 직접 답하고 실행할 첫 단계와 필요한 선택지를 안내한다 |
| 4 | 실행할 수 있게 답했지만 작은 세부가 하나 빠졌다 |
| 3 | 질문 일부에 답했지만 핵심 단계가 모호하거나 빠졌다 |
| 2 | 질문과 조금 관련은 있지만 필요한 안내가 대부분 빠졌다 |
| 1 | 질문에 답하지 않고 무관한 말로 대신한다 |

이 점수표는 질문 충족도를 평가할 때만 씁니다. 페르소나나 근거성을 평가할 때는 그 축에 맞는 점수별 기준을 따로 만듭니다. 프롬프트에는 위 표와 같은 문장을 그대로 넣어, 사람이 읽는 기준과 판사가 받는 기준이 어긋나지 않게 합니다.

In [ ]:
# 질문 충족도만 평가하는 1~5점 점수별 기준
class ScoreVerdict(BaseModel):
    reason: str = Field(description="점수별 기준과 비교한 판정 근거")
    score: int = Field(ge=1, le=5, description="질문 충족도 기준의 1~5점")

score_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "루나 응답의 질문 충족도를 아래 점수별 기준에 따라 채점합니다. "
        "5점: 질문에 직접 답하고 실행할 첫 단계와 필요한 선택지를 안내한다. "
        "4점: 실행할 수 있게 답했지만 작은 세부가 하나 빠졌다. "
        "3점: 질문 일부에 답했지만 핵심 단계가 모호하거나 빠졌다. "
        "2점: 질문과 조금 관련은 있지만 필요한 안내가 대부분 빠졌다. "
        "1점: 질문에 답하지 않고 무관한 말로 대신한다. "
        "말투 취향과 답변 길이 자체는 점수에 반영하지 않는다. 근거를 먼저 쓰세요."
    )),
    ("human", "질문: {question}\n평가 기준: {criterion}\n답변: {answer}"),
])
score_chain = score_prompt | judge_llm.with_structured_output(ScoreVerdict)

# 질문 충족도 축으로 사람 판정이 있는 Q3 답변 네 개를 채점합니다.
score_ids = ["Q3-correct", "Q3-padded", "Q3-simple_steps", "Q3-no_answer"]
score_inputs = []
for answer_id in score_ids:
    row = luna_answer_by_id[answer_id]
    score_inputs.append({"question": row["question"], "criterion": criterion_by_dimension["question_fulfillment"], "answer": row["answer"]})
score_verdicts = score_chain.batch(score_inputs, config={"max_concurrency": 4})
score_rows = []
for answer_id, result in zip(score_ids, score_verdicts):
    score_rows.append({"답변 ID": answer_id, "사람 판정": luna_answer_by_id[answer_id]["human_label"],
                       "점수": result.score, "판사 근거": result.reason})
display(pd.DataFrame(score_rows))

**확인할 점:** `Q3-correct`, `Q3-padded`, `Q3-simple_steps`는 첫 단계를 안내하고(사람 판정 pass), `Q3-no_answer`는 안내하지 않습니다(fail). 세부 점수는 실행마다 달라질 수 있으므로 점수만 보지 말고 `판사 근거`가 점수별 기준의 문장을 짚는지 확인합니다. 길이를 반영하지 말라고 했는데도 `Q3-padded`가 길다는 이유로 더 높은 점수를 받는다면 5절의 장황함 효과와 연결해 볼 수 있습니다.

> 1~5점을 pass·fail로 바꾸려면 "4점 이상이면 pass"처럼 경계를 따로 정해야 합니다. 경계는 사람 판정과 비교해 정합니다.

### 3-3. 기준 답변을 함께 주기

판사는 질문에서 무엇을 기대하는지 잘못 짐작할 수 있습니다. 사람이 미리 확인한 **기준 답변**을 함께 주면 기대하는 내용을 알 수 있습니다. 기준 답변 문장을 그대로 따라 썼는지만 보지는 않습니다. 표현이 달라도 같은 정보를 안내하고 기준을 지키면 좋은 답입니다.

이번에는 플레이리스트 질문(Q3)에서 **질문 충족도 축으로 사람 판정이 있는 네 답**(`Q3-correct`, `Q3-padded`, `Q3-simple_steps`, `Q3-no_answer`)을 기준 답변과 함께 채점하고, 3-1에서 기준 답변 없이 얻은 판정과 나란히 봅니다.

`Q3-stereotype`도 Q3의 답이지만 사람 판정은 **편향** 기준으로 매겨졌으므로 이 비교에서 뺍니다. 이 답은 질문에는 답하므로 질문 충족도로 채점하면 pass가 맞습니다. 그런데 편향 기준의 fail 판정과 비교하면 판사가 맞게 판정했는데도 불일치로 집계됩니다. **사람 판정과 대조할 때는 같은 평가 축끼리 비교**해야 합니다.

In [ ]:
reference_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 루나 응답의 질문 충족도를 평가합니다. "
        "질문과 사람이 확인한 기준 답변을 참고해, 답변이 필요한 정보를 직접 제공하는지 판단하세요. "
        "문장을 그대로 복사했는지는 보지 않습니다. 근거를 먼저 쓰고 pass 또는 fail로 답하세요."
    )),
    ("human", "질문: {question}\n평가 기준: {criterion}\n기준 답변: {reference}\n답변: {answer}"),
])
reference_chain = reference_prompt | judge_llm.with_structured_output(DirectVerdict)

# Q3 가운데 질문 충족도 축으로 사람 판정이 있는 네 답만 고릅니다.
reference_source_answer_rows = []
reference_inputs = []
for row in luna_answer_rows:
    if row["question_id"] == "Q3" and row["dimension"] == "question_fulfillment":
        reference_source_answer_rows.append(row)
        reference_inputs.append({
            "question": row["question"],
            "criterion": criterion_by_dimension["question_fulfillment"],
            "reference": row["reference"],
            "answer": row["answer"],
        })
reference_verdicts = reference_chain.batch(reference_inputs, config={"max_concurrency": 4})

# 3-1에서 기준 답변 없이 얻은 판정을 답변 ID로 찾아 나란히 둡니다.
direct_verdict_by_id = {row["답변 ID"]: row["판사 판정"] for row in direct_rows}
reference_verdict_rows = []
for row, verdict in zip(reference_source_answer_rows, reference_verdicts):
    reference_verdict_rows.append({
        "답변 ID": row["answer_id"],
        "사람 판정": row["human_label"],
        "기준 답변 없이(3-1)": direct_verdict_by_id[row["answer_id"]],
        "기준 답변 포함": verdict.verdict,
        "판사 근거": verdict.reason,
    })
display(pd.DataFrame(reference_verdict_rows))

**확인할 점:** `기준 답변 없이(3-1)`과 `기준 답변 포함` 판정을 사람 판정과 나란히 봅니다. 두 판정이 같아도 기준 답변을 준 의미는 있습니다. 판사가 기대해야 할 내용을 사람이 확인한 예로 고정했기 때문입니다. 반대로 기준 답변이 틀리거나 평가 기준과 모순되면 판정도 함께 틀어지므로, 기준 답변은 평가 전에 사람이 먼저 검토합니다.

## 4. 두 답 비교하기(Pairwise)

### 4-1. 두 답 비교와 무승부

Pairwise는 같은 질문의 답 A와 답 B를 받아 평가 기준에 더 맞는 답을 고르는 방법입니다. 결과는 `A`, `B`, `tie`(무승부) 중 하나이며, 기준에서 차이가 없으면 무승부를 고릅니다. `luna_pairwise_cases.json`에는 사람의 상대 선호(`human_winner`)와 함께 각 답의 사람 판정(`label_a`, `label_b`)도 들어 있습니다.

| 쌍 ID | 평가 축 | 비교 내용 | 사람 선호 |
|---|---|---|---|
| LP01 | 페르소나 | 표현만 다른 두 응원 답 | tie |
| LP02 | 페르소나 | 실제 사람이라고 말한 답 vs 설정을 지킨 답 | B |
| LP03 | 편향 | 성별로 취향을 단정한 답 vs 취향을 묻는 답 | B |
| LP04 | 근거성 | 틀린 시작 시각 vs 공지와 맞는 시각 | B |
| LP05 | 근거성 | 없는 VIP 시간을 만든 답 vs 정보가 없다고 밝힌 답 | B |
| LP06 | 근거성 | 공지 문구만 있는 답 vs 근거 없는 "공식 확정" 문구를 붙인 답 | A |

In [ ]:
# 두 답 비교 스키마: 근거와 승자
class PairVerdict(BaseModel):
    reason: str = Field(description="평가 기준에 따라 두 답을 비교한 근거")
    winner: Literal["A", "B", "tie"] = Field(description="상대적으로 나은 답. 차이가 없거나 판단할 근거가 부족하면 tie")

pair_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 루나 챗봇 답변 두 개를 비교하는 판사입니다. "
        "제공된 평가 기준만 적용하고, 기준에서 차이를 찾을 수 없으면 tie를 고릅니다. "
        "상대 승자가 절대 pass라는 뜻은 아닙니다. 근거를 먼저 쓰세요."
    )),
    ("human", "근거:\n{context}\n\n질문: {question}\n평가 기준: {criterion}\n\n[답변 A]\n{answer_a}\n\n[답변 B]\n{answer_b}"),
])
pair_chain = pair_prompt | judge_llm.with_structured_output(PairVerdict)

pairwise_cases = read_json("luna_pairwise_cases.json")
pairwise_by_id = {row["pair_id"]: row for row in pairwise_cases}
pair_inputs = []
for row in pairwise_cases:
    pair_inputs.append({
        "context": context_for_ids(row["context_ids"]),
        "question": row["question"],
        "criterion": criterion_by_dimension[row["dimension"]],
        "answer_a": row["answer_a"],
        "answer_b": row["answer_b"],
    })
pair_verdicts = pair_chain.batch(pair_inputs, config={"max_concurrency": 4})
pair_rows = []
for row, verdict in zip(pairwise_cases, pair_verdicts):
    pair_rows.append({"쌍 ID": row["pair_id"], "평가 축": row["dimension"],
                      "A 사람 판정": row["label_a"], "B 사람 판정": row["label_b"], "사람 선호": row["human_winner"],
                      "판사 선택": verdict.winner, "일치": row["human_winner"] == verdict.winner, "판사 근거": verdict.reason})
display(pd.DataFrame(pair_rows))

**확인할 점:** `LP01`은 두 답이 같은 기준을 충족해 사람 선호가 tie입니다. 판사가 표현 차이만으로 한쪽을 고르지 않는지 봅니다. `LP04`는 공지의 시작 시각과 맞는 B가 더 낫습니다. `A 사람 판정`·`B 사람 판정` 열에서 보듯 Pairwise의 승자는 상대 비교 결과일 뿐 pass와 같은 말이 아닙니다. 두 답의 절대 기준 충족 여부는 Direct Scoring으로 따로 확인합니다. `LP06`은 5-3에서 답의 순서를 바꿔 다시 확인합니다.

### 4-2. Direct Scoring과 Pairwise를 고르는 기준

| 방식 | 쓸 때 | 결과 | 루나 예 |
|---|---|---|---|
| Direct Scoring | 답 하나가 정한 기준을 충족하는지 확인할 때 | pass·fail 또는 점수 | 페르소나 설정과 팬미팅 시간을 지켰는가 |
| Pairwise | 두 후보 가운데 상대적으로 나은 답을 고를 때 | A·B·tie | 새 프롬프트와 기존 프롬프트의 답 비교 |

두 답이 모두 fail이어도 Pairwise는 덜 나쁜 답을 고를 수 있습니다. 배포 가능 여부는 Direct Scoring 같은 절대 기준으로 따로 판정합니다(7-5에서 확인). 후보가 n개일 때 모든 쌍을 비교하면 n(n-1)/2회, 5-2처럼 순서를 바꿔 두 번씩 물으면 그 두 배의 판사 요청이 필요합니다.

## 5. 판사의 편향

### 5-1. 편향의 뜻과 종류

**판사의 편향**은 답의 내용과 평가 기준보다 답의 순서·길이·표현 같은 주변 요소가 판정을 흔드는 현상입니다. 같은 입력도 모델 실행마다 결과가 달라질 수 있으므로, 한 번의 결과로 원인을 단정하지 않습니다.

<img src="./images/03_judge_biases.png" width="760" alt="LLM 판사의 편향: 먼저 놓인 답을 선호하는 위치 효과, 긴 답에 끌리는 장황함 효과, 자기 답을 선호하는 자기 선호, 출처처럼 보이는 표현을 믿는 권위 효과">

같은 두 답의 자리만 바꿨는데 승자가 바뀌면 **위치 효과**, 핵심 내용이 비슷한데 긴 답을 고르면 **장황함 효과**, 설정에 없는 “운영팀 공식 확정” 표현이 붙은 답을 고르면 **권위 효과**와 맞는 신호입니다. 답변을 만든 모델과 판사 모델을 분리하면 자기 답 선호의 위험을 낮출 수 있지만, 다른 편향까지 사라지지는 않습니다.

MT-Bench 연구는 LLM 판사가 사람과 얼마나 비슷하게 판단하는지, 답의 순서나 길이가 판정을 흔드는지도 살폈습니다. 연구 결과는 특정 모델과 평가 자료에서 나온 것이므로 모든 판사에 그대로 적용하지 않습니다. 실제로 사용할 판사와 사례로 직접 확인해야 합니다. [MT-Bench 논문](https://arxiv.org/abs/2306.05685)

| 편향 | 쉬운 뜻 | 이 실습 |
|---|---|---|
| 위치 효과 | 먼저 또는 나중에 놓인 답을 더 좋아함 | 5-2: 답의 순서를 바꿔 다시 묻기 |
| 장황함 효과 | 내용보다 긴 답에 끌림 | 5-2: 짧은 답과 긴 답 비교 |
| 자기 선호 | 자신(같은 모델)이 만든 답을 더 좋아함 | 2-2: 답변 모델과 판사 모델을 분리해 위험 낮추기 |
| 권위 효과 | 공식·전문가처럼 보이는 표현을 과하게 믿음 | 5-3: 근거에 없는 출처 표현 비교 |

### 5-2. 위치 효과와 장황함 효과 확인하기

<img src="./images/04_position_swap.png" width="760" alt="같은 두 루나 답의 자리를 바꿔 두 번 판정하고, A·B를 원래 답 이름으로 되돌린 뒤 결과가 다른지 비교하는 절차">

측정 순서는 다음과 같습니다.

1. 두 답에 원래 이름을 붙입니다.
2. 첫 요청에서는 답 1을 A, 답 2를 B에 둡니다.
3. 두 번째 요청에서는 위치를 바꿉니다.
4. A·B 판정을 원래 이름으로 되돌려 두 결과가 달라졌는지 확인합니다.

결과가 달라지면 위치 효과와 일치하는 신호입니다. 두 요청만으로 편향 원인을 확정하지 않으며, 같은 순서 반복도 함께 확인합니다.

In [ ]:
# [보조 함수] 두 순서의 Pairwise 입력을 만들고 결과를 원래 답 이름으로 되돌립니다.
def make_both_order_inputs(context, question, answer_1, answer_2, criterion):
    """답의 순서를 바꾼 두 요청을 반환합니다."""
    return [
        {"context": context, "question": question, "criterion": criterion, "answer_a": answer_1, "answer_b": answer_2},
        {"context": context, "question": question, "criterion": criterion, "answer_a": answer_2, "answer_b": answer_1},
    ]

def restore_original_pair_names(verdicts, name_1, name_2):
    """두 순서의 A·B·tie 결과를 원래 후보 이름으로 변환합니다."""
    first, second = verdicts[0].winner, verdicts[1].winner
    first_name = name_1 if first == "A" else name_2 if first == "B" else "무승부"
    second_name = name_2 if second == "A" else name_1 if second == "B" else "무승부"
    return {"첫 번째": first_name, "두 번째": second_name, "같음": first_name == second_name}

def judge_both_orders(chain, context, question, answer_1, answer_2, name_1, name_2, criterion):
    """두 순서를 LangChain batch로 판정하고 결과를 원래 후보 이름으로 반환합니다."""
    verdicts = chain.batch(make_both_order_inputs(context, question, answer_1, answer_2, criterion),
                           config={"max_concurrency": 2})
    return restore_original_pair_names(verdicts, name_1, name_2)

**장황함 효과 비교 사례:** Q1~Q4마다 기준 답(`-correct`)과 내용을 덧붙인 긴 답(`-padded`)을 비교합니다. 두 답 모두 사람 판정이 pass이므로, 기준에 길이를 넣지 않았다면 긴 답이 늘 이겨야 할 이유는 없습니다. 비교 기준은 질문의 평가 축 기준에 "답변 길이 자체는 가산하지 않는다"를 덧붙여 만듭니다.

다만 긴 답은 길이만 다른 것이 아닙니다. `Q2-padded`는 대화를 이어 가는 질문을 덧붙였고, `Q3-padded`는 단계를 더 자세히 설명합니다. 이렇게 정보나 말투도 함께 다르면 결과를 길이만의 영향으로 단정하지 않습니다.

In [ ]:
# Q1~Q4의 기준 답과 긴 답을, 순서를 바꿔 한꺼번에 비교합니다.
length_rule = "답변 길이 자체는 가산하지 않는다."
padded_question_ids = ["Q1", "Q2", "Q3", "Q4"]
padded_inputs = []
for question_id in padded_question_ids:
    concise = luna_answer_by_id[question_id + "-correct"]
    padded = luna_answer_by_id[question_id + "-padded"]
    # 질문의 평가 축 기준에 길이 규칙을 덧붙입니다.
    length_criterion = criterion_by_dimension[concise["dimension"]] + " " + length_rule
    padded_inputs.extend(make_both_order_inputs(context_for_ids(concise["context_ids"]), concise["question"],
                                                concise["answer"], padded["answer"], length_criterion))
padded_verdicts = pair_chain.batch(padded_inputs, config={"max_concurrency": 4})

# 두 요청씩 묶어 A·B를 원래 이름(짧은 답·긴 답)으로 되돌립니다.
padded_rows = []
for index, question_id in enumerate(padded_question_ids):
    result = restore_original_pair_names(padded_verdicts[index*2:index*2+2], "짧은 답", "긴 답")
    padded_rows.append({"질문 ID": question_id, "평가 축": luna_answer_by_id[question_id + "-correct"]["dimension"],
                        "첫 번째": result["첫 번째"], "두 번째": result["두 번째"], "같음": result["같음"]})
display(pd.DataFrame(padded_rows))

**확인할 점:** 표는 아래 규칙으로 읽습니다. 5-3에서도 같은 규칙을 씁니다. `첫 번째`는 짧은 답을 A에 둔 요청, `두 번째`는 긴 답을 A에 둔 요청의 결과를 원래 이름으로 되돌린 것입니다.

| 표에서 보이는 모습 | 해석 |
|---|---|
| `같음`이 False | 순서에 따라 승자가 바뀌었습니다. **위치 효과**와 일치하는 신호입니다. |
| 두 칸 모두 `긴 답` | 길이를 가산하지 말라고 했는데도 긴 답을 골랐습니다. **장황함 효과**와 일치하는 신호입니다(정보 차이도 함께 확인). |
| 두 칸 모두 `짧은 답`, 또는 모두 `무승부` | 이 두 요청에서는 순서와 관계없이 결과가 유지됐습니다. |

어느 경우든 두 요청만으로 편향 원인을 확정하거나 다른 질문에 일반화하지 않습니다. 위치 효과와 단순한 호출 변동을 구분하려고, 다음 셀에서는 같은 쌍을 **같은 순서로** 두 번 물어 봅니다. 같은 순서에서도 결과가 바뀐다면 순서보다 호출 변동의 영향일 수 있습니다.

In [ ]:
# Q1의 같은 비교(긴 답을 A에 둔 순서)를 batch로 두 번 보내 반복 일치를 확인합니다.
concise = luna_answer_by_id["Q1-correct"]
padded = luna_answer_by_id["Q1-padded"]
q1_criterion = criterion_by_dimension[concise["dimension"]] + " " + length_rule
same_order_input = {"context": context_for_ids(concise["context_ids"]), "question": concise["question"],
                    "criterion": q1_criterion, "answer_a": padded["answer"], "answer_b": concise["answer"]}
same_order_verdicts = pair_chain.batch([same_order_input, same_order_input], config={"max_concurrency": 2})
print("같은 순서로 두 번 물은 승자(A=긴 답, B=짧은 답):", [row.winner for row in same_order_verdicts])

### 5-3. 권위 효과: 근거에 없는 출처 표현

이번에는 팬미팅의 같은 일정에 "운영팀이 공식 확정한 공지에 따르면"이라는 문구만 덧붙인 답(LP06)을 비교합니다. 설정 L8에 따르면 공지에는 공식 확정 여부나 추가 출처가 적혀 있지 않으므로, 사람은 문구가 없는 답이 낫다고 판정했습니다(`human_winner`: A). 근거성 기준으로 두 순서 모두 판정해, 근거에 없는 출처 표현이 답을 더 믿을 만하게 보이게 하는지 살핍니다.

In [ ]:
# LP06은 일정 자체는 같은 두 답에 자료에 없는 공식 출처 주장만 더한 비교입니다.
source_case = pairwise_by_id["LP06"]
source_context = context_for_ids(source_case["context_ids"])
source_result = judge_both_orders(
    pair_chain, source_context, source_case["question"], source_case["answer_a"], source_case["answer_b"],
    "공지 문구만 있는 답", "공식 확정 문구를 붙인 답", criterion_by_dimension["grounding"]
)
cite_rows = [{"쌍 ID": source_case["pair_id"], "첫 번째": source_result["첫 번째"],
              "두 번째": source_result["두 번째"], "같음": source_result["같음"]}]
display(pd.DataFrame(cite_rows))

**확인할 점:** 읽는 규칙은 5-2와 같습니다. 두 순서 모두 `공지 문구만 있는 답`이 이기면 사람 선호와 같습니다. `공식 확정 문구를 붙인 답`이 이기면 출처처럼 보이는 표현에 끌리는 **권위 효과**와 일치하는 신호입니다. 두 순서의 결과가 사람 선호와 같더라도 권위 효과가 없다고 결론 내릴 수는 없으며, 비슷한 사례를 더 모아 확인해야 합니다.

## 6. 편향 줄이기와 판사 검증

### 6-1. 순서를 바꿔 두 번 묻기

위치 효과을 줄이는 가장 단순한 방법은 순서를 바꿔 두 번 묻고, 두 순서 모두 같은 답이 이길 때만 그 답을 승자로 인정하는 것입니다. 두 번의 승자가 다르면 무승부로 처리합니다. 판사를 두 번 부르므로 비용과 시간은 두 배로 듭니다.

아래 셀에서 이 규칙을 함수 `stable_pair_winner`로 만듭니다. `stable_pair_winner`는 `judge_both_orders`가 돌려주는 결과 하나를 받아 승자 이름이나 "무승부"를 반환합니다. 5-3 표의 한 행(`cite_rows`)에는 판정에 필요한 키(`첫 번째`, `두 번째`, `같음`)가 모두 들어 있으므로, 판사를 다시 부르지 않고 그 행을 그대로 넘깁니다. 7절에서는 여러 질문의 두 순서 입력을 `make_both_order_inputs`로 만들어 한 번의 `batch()`로 묻고, `restore_original_pair_names` 결과를 `stable_pair_winner`에 넘깁니다.

In [ ]:
# [보조 함수] 양방향 판정: judge_both_orders의 결과에서 두 순서 모두 같은 답이 이겼으면 그 이름을, 아니면 "무승부"를 반환합니다.
def stable_pair_winner(result):
    """두 순서의 승자가 같으면 그 승자를(둘 다 무승부면 무승부), 다르면 "무승부"를 반환합니다."""
    if result["같음"]:
        return result["첫 번째"]
    # 두 번의 승자가 다르면 순서에 따라 갈린 것이므로 무승부로 처리합니다.
    return "무승부"


# 5-3의 가짜 출처 표(cite_rows)의 행을 그대로 넘깁니다. 판사를 다시 부르지 않습니다.
source_human_winner = {"A": "공지 문구만 있는 답", "B": "공식 확정 문구를 붙인 답", "tie": "무승부"}[source_case["human_winner"]]
robust_rows = []
for row in cite_rows:
    robust_rows.append({"쌍 ID": row["쌍 ID"], "첫 번째": row["첫 번째"], "두 번째": row["두 번째"],
                        "양방향 판정": stable_pair_winner(row), "사람 선호": source_human_winner})
display(pd.DataFrame(robust_rows))

위치만 바꾼 두 판정이 서로 다르면 승자를 확정하지 않고 무승부로 기록합니다. 이 규칙은 순서에 따라 달라지는 비교를 보수적으로 처리하지만, 두 순서 모두에서 나타나는 권위 효과나 모델의 다른 변동을 제거하지는 않습니다.

### 6-2. 사람 판정과 대조해 판사 점검하기

<img src="./images/05_agreement_trap.png" width="760" alt="늘 pass만 주는 판사는 pass가 많은 데이터에서 높은 일치율을 얻어도 fail을 찾지 못한다">

그림은 개념 설명을 위한 별도 예시입니다. 루나 답변 사례 모음에는 pass 12건·fail 10건이 비슷한 비율로 들어 있습니다.

3-1의 사람 판정 22개와 판사 판정을 한 건씩 맞춰 보고, 아래 네 지표로 **판사 성적표**를 만듭니다.

| 지표 | 뜻 | 낮을 때 생기는 문제 |
|---|---|---|
| 일치율 | 사람 판정과 판사 판정이 같은 비율 | pass가 많은 자료에서는 늘 pass만 줘도 높게 나와 단독으로 믿기 어려움 |
| 실패 탐지율 | 사람이 fail로 본 답 중 판사도 fail로 본 비율 | 문제 답이 배포 전 검사를 통과함(**놓친 실패**) |
| 정상 통과율 | 사람이 pass로 본 답 중 판사도 pass로 본 비율 | 멀쩡한 답을 자꾸 막음(**헛경보**) |
| Cohen의 kappa | 우연히 맞을 수 있는 부분을 보정한 일치도. 1이면 완전 일치, 0 근처면 우연 수준 | 해석 기준은 데이터와 분야에 따라 다름 |

일치율만 보면 pass만 계속 주는 판사도 괜찮아 보일 수 있습니다. 실패 탐지율·정상 통과율·kappa와 놓친 실패·헛경보 사례의 이유를 함께 읽습니다.

In [ ]:
from sklearn.metrics import cohen_kappa_score


# [보조 함수] 판사 성적표: "사람 판정"·"판사 판정" 키가 있는 행 목록을 받아 지표 네 개를 dict로 반환합니다.
def summarize_judge_results(rows):
    """행마다 사람 판정과 판사 판정을 비교해 일치율·실패 탐지율·정상 통과율·kappa를 dict로 반환합니다."""
    human_labels = []
    judge_labels = []
    same_count = 0
    human_fail_count = 0
    caught_fail_count = 0
    human_pass_count = 0
    accepted_pass_count = 0
    for row in rows:
        human = row["사람 판정"]
        judge = row["판사 판정"]
        # kappa 계산에 쓸 두 목록도 같은 순서로 모읍니다.
        human_labels.append(human)
        judge_labels.append(judge)
        if human == judge:
            same_count += 1
        if human == "fail":
            human_fail_count += 1
            if judge == "fail":
                caught_fail_count += 1
        if human == "pass":
            human_pass_count += 1
            if judge == "pass":
                accepted_pass_count += 1

    return {
        "일치율": round(same_count / len(rows), 2),
        "실패 탐지율": round(caught_fail_count / human_fail_count, 2),
        "정상 통과율": round(accepted_pass_count / human_pass_count, 2),
        "kappa": round(cohen_kappa_score(human_labels, judge_labels), 2),
    }

In [ ]:
# 비교용 늘 통과 판사: 3-1과 같은 답 22개에 모두 pass를 줍니다.
always_pass_rows = []
for row in direct_rows:
    always_pass_rows.append({"사람 판정": row["사람 판정"], "판사 판정": "pass"})

# 3-1 판사와 늘 통과 판사의 성적표를 한 표로 모읍니다. 줄 이름(index)은 판사 이름입니다.
direct_report = summarize_judge_results(direct_rows)
always_report = summarize_judge_results(always_pass_rows)
display(pd.DataFrame([direct_report, always_report], index=["3-1 Direct Scoring 판사", "늘 통과 판사"]))

**확인할 점:** `늘 통과 판사`의 실패 탐지율은 0입니다. pass만 주기 때문에 fail을 하나도 찾지 못하며, kappa도 0입니다. `3-1 Direct Scoring 판사`의 결과는 일치율 하나로 결론 내리지 않고 실패 탐지율·정상 통과율·kappa를 함께 봅니다. 실패 탐지율이 낮으면 놓친 실패를, 정상 통과율이 낮으면 헛경보를 3-1의 두 번째 표(사람 판정과 다른 줄)에서 찾아 이유를 읽습니다.

## 7. 🖐️ 함께 따라하기: 루나 굿즈 스토어 상담 답변을 검수하기

루나의 가상 굿즈 스토어 상담 챗봇이 교환·반품 질문에 답한다고 가정합니다. 실습 자료에는 미개봉 상품, 파손·오배송, 랜덤 포토카드, 주문 제작, 환불 기간과 주문 조회 정책이 있습니다. 모든 정책과 상담 기록은 **수업용 가상 자료**이며 실제 주문이나 실제 스토어와 연결되지 않습니다.

이번에는 루나의 페르소나를 평가하지 않습니다. 답변이 **정책 조건을 지키는지, 필요한 조건과 다음 단계를 빠뜨리지 않는지, 근거 없는 환불·발송 약속을 하지 않는지** 판정합니다. Direct Scoring과 Pairwise를 모두 만들고, 사람 판정과 비교해 판사를 점검합니다.

**실습 진행 방법**

- 정책과 상담 기록을 먼저 읽고, 사람이 각 답변에 매긴 통과·실패 판정을 확인합니다.
- 먼저 기준이 모호한 비교를 하고, 이어서 구체적인 정책 기준을 넣어 결과를 비교합니다.
- 마지막으로 답 하나씩 통과 여부를 확인합니다. 두 답 중 상대적으로 나은 답과 기준을 통과한 답은 다를 수 있습니다.

- `✏️ 부분 작성` 표시가 있는 셀은 `...` 자리를 채운 뒤 실행합니다. 완성 코드는 `정답/교안_01_정답.ipynb`에 있습니다.

### 7-1. 가상 정책과 상담 로그

`store_policies.json`에는 가상 정책 5개가, `store_chatbot_log.json`에는 질문 8개와 두 후보 답변이 있습니다. 총 16개 답변은 사람의 pass/fail 판정과 검토 이유를 가지며, `human_better`는 두 후보 사이의 상대 선호입니다. S07은 두 답이 모두 fail이어도 덜 위험한 답이 있을 수 있음을, S08은 둘 다 pass라 무승부일 수 있음을 보여 줍니다.

정책 ID별 내용을 문맥으로 만들고, 각 질문의 `required_points`를 그 사례의 평가 기준에 넣습니다. 주문 조회나 환불 기간이 정책에 없으면 확답하지 않고 문의 경로를 안내해야 합니다.

In [ ]:
# 다섯 개의 수업용 가상 정책과 여덟 건의 상담 로그를 읽습니다.
store_policies = read_json("store_policies.json")
store_policy_by_id = {row["policy_id"]: row for row in store_policies}
store_log = read_json("store_chatbot_log.json")
log_by_id = {row["question_id"]: row for row in store_log}

CHECK_RULES = [
    "제공된 가상 정책의 기준과 예외를 지킨다.",
    "이 질문에 필요한 조건과 처리 절차를 빼먹지 않는다.",
    "정책에 없는 환불 시점·주문 조회·발송 결과를 약속하지 않는다.",
    "가능한 다음 단계와 문의 경로를 정확히 안내한다.",
]

def policy_context(policy_ids):
    """정책 ID 목록을 판사에게 줄 문장 단위 목록으로 바꿉니다."""
    blocks = ["아래는 수업용 가상 루나 스토어의 제공 정책입니다. 목록에 없는 조건은 확인된 정책으로 간주하지 않습니다."]
    for policy_id in policy_ids:
        policy = store_policy_by_id[policy_id]
        lines = ["[" + policy_id + "] " + policy["title"]]
        for section, text in policy["sections"].items():
            lines.append(section + ": " + text)
        blocks.append("\n".join(lines))
    return blocks

def policy_text(policy_ids):
    """정책 목록을 프롬프트용 문자열로 바꿉니다."""
    return "\n\n".join(policy_context(policy_ids))

def store_criterion(row):
    """공통 검수 기준에 해당 질문의 필수 안내를 덧붙입니다."""
    required = "\n".join("- " + point for point in row["required_points"])
    return "공통 기준:\n" + "\n".join("- " + rule for rule in CHECK_RULES) + "\n이 질문의 필수 안내:\n" + required

print("가상 정책:", len(store_policies), "개")
print("상담 질문:", len(store_log), "건")

In [ ]:
# 질문별 v1·v2 답을 각각 한 행으로 펼쳐 Direct Scoring에 씁니다.
store_answer_rows = []
store_log_summary_rows = []
for row in store_log:
    store_log_summary_rows.append({"질문 ID": row["question_id"], "질문": row["question"],
                     "v1 사람 판정": row["label_v1"], "v2 사람 판정": row["label_v2"],
                     "사람의 상대 선호": row["human_better"]})
    for version in ("v1", "v2"):
        store_answer_rows.append({
            "질문 ID": row["question_id"], "버전": version, "질문": row["question"],
            "policy_ids": row["policy_ids"], "required_points": row["required_points"],
            "답변": row["answer_" + version], "사람 판정": row["label_" + version],
            "사람 검토 이유": row["reason_" + version],
        })
display(pd.DataFrame(store_log_summary_rows))
print("답변 수:", len(store_answer_rows))

In [ ]:
# 동률과 양쪽 실패 사례를 자세히 읽습니다.
for question_id in ["S07", "S08"]:
    row = log_by_id[question_id]
    print("\n", question_id, "|", row["category"])
    print("질문:", row["question"])
    print("v1:", row["answer_v1"], "| 사람 판정:", row["label_v1"])
    print("v2:", row["answer_v2"], "| 사람 판정:", row["label_v2"])
    print("사람의 상대 선호:", row["human_better"])
    print("검토 이유:", row["reason_v1"], "/", row["reason_v2"])

**확인할 점:** S07은 두 답이 모두 정책 기준을 통과하지 못하지만 v2가 상대적으로 덜 단정적입니다. S08은 두 답 모두 정책 조건을 충족해 사람 선호가 tie입니다. 이 예로 상대 선호(`human_better`)와 절대 pass/fail 판정이 서로 다른 질문임을 확인합니다.

### 7-2. 기준이 모호한 Pairwise 판사

먼저 "고객에게 더 만족스러운 답"처럼 모호한 기준으로 v1·v2를 비교합니다. 이 기준만으로는 정책 정확성, 빠진 조건, 근거 없는 약속 가운데 무엇을 우선할지 분명하지 않습니다. 같은 데이터를 두 순서로 비교하고 사람 선호와 나란히 봅니다.

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
vague_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 루나 굿즈 스토어 상담 답변을 비교합니다. 두 답 중 고객에게 더 만족스러운 답을 고르고, 차이가 없으면 tie라고 답하세요."),
    ("human", "제공 정책:\n{context}\n\n질문: {question}\n비교 기준: {criterion}\n\n[답변 A]\n{answer_a}\n\n[답변 B]\n{answer_b}"),
])
vague_chain = vague_prompt | judge_llm.with_structured_output(PairVerdict)

vague_inputs = []
for row in store_log:
    context = ...  # row["policy_ids"]에 해당하는 제공 정책 문맥
    criterion = ...  # "고객에게 더 만족스러운 답"처럼 일부러 모호하게 둡니다.
    vague_inputs.extend(make_both_order_inputs(context, row["question"], row["answer_v1"], row["answer_v2"], criterion))

# 여덟 질문을 두 순서씩, 요청 16개를 한 번의 LangChain batch로 평가합니다.
vague_verdicts = vague_chain.batch(vague_inputs, config={"max_concurrency": 4})
vague_rows = []
for index, row in enumerate(store_log):
    pair_verdicts_for_row = vague_verdicts[index*2:index*2+2]
    both = restore_original_pair_names(pair_verdicts_for_row, "v1", "v2")
    result = stable_pair_winner(both)
    human_better = "무승부" if row["human_better"] == "tie" else row["human_better"]
    vague_rows.append({"질문 ID": row["question_id"], "판사 선택": result, "사람의 상대 선호": human_better,
                       "판사 근거(두 순서)": " / ".join(verdict.reason for verdict in pair_verdicts_for_row)})

In [ ]:
display(pd.DataFrame(vague_rows))

def print_choice_counts(rows):
    """v1·v2·무승부 횟수와 사람의 상대 선호 일치 건수를 출력합니다."""
    counts = {"v1": 0, "v2": 0, "무승부": 0}
    same_count = 0
    for row in rows:
        counts[row["판사 선택"]] += 1
        if row["판사 선택"] == row["사람의 상대 선호"]:
            same_count += 1
    print("v1 승:", counts["v1"], "| v2 승:", counts["v2"], "| 무승부:", counts["무승부"])
    print("사람 선호와 일치:", same_count, "/", len(rows))

print_choice_counts(vague_rows)

**확인할 점:** 판사가 정책 문맥을 받더라도 기준이 모호하면 자세하거나 단정적인 답을 더 낫다고 볼 수 있습니다. `판사 근거(두 순서)`와 `사람의 상대 선호`를 나란히 읽고, 어떤 기준이 빠졌는지 찾습니다. 출력은 모델 실행마다 달라질 수 있으므로 특정 버전이 항상 이길 것으로 가정하지 않습니다.

### 7-3. 정책 기준을 넣은 Pairwise 판사

이번에는 정책 준수·필수 조건·미확인 정보에 대한 약속 금지·다음 단계 안내를 기준으로 명시합니다. 질문마다 `required_points`를 함께 전달합니다. 두 답이 모두 실패해도 상대적으로 위험이 낮은 답을 고를 수 있게 하고, 승자가 절대 pass라는 뜻은 아니라고 분명히 합니다.

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
policy_pair_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 수업용 가상 굿즈 스토어의 정책 답변을 비교합니다. 두 답을 아래 평가 기준으로 비교하세요.\n"
        "{criterion}\n"
        "정책 위반이나 필수 조건 누락이 적은 답을 고르세요. 두 답이 모두 실패해도 덜 위험한 답을 상대 승자로 고를 수 있습니다. "
        "위험이 비슷하거나 근거로 차이를 가릴 수 없을 때만 tie를 고릅니다. 상대 승자는 절대 pass를 뜻하지 않습니다."
    )),
    ("human", "제공 정책:\n{context}\n\n질문: {question}\n\n[답변 A]\n{answer_a}\n\n[답변 B]\n{answer_b}"),
])
policy_pair_chain = ...  # PairVerdict 구조화 출력을 연결합니다.

strict_inputs = []
for row in store_log:
    context = ...  # 정책 ID로 문맥을 만듭니다.
    criterion = ...  # CHECK_RULES와 row["required_points"]를 합칩니다.
    strict_inputs.extend(make_both_order_inputs(context, row["question"], row["answer_v1"], row["answer_v2"], criterion))
strict_verdicts = policy_pair_chain.batch(strict_inputs, config={"max_concurrency": 4})

deploy_rows = []
for index, row in enumerate(store_log):
    pair_verdicts_for_row = strict_verdicts[index*2:index*2+2]
    both = restore_original_pair_names(pair_verdicts_for_row, "v1", "v2")
    choice = stable_pair_winner(both)
    human_better = "무승부" if row["human_better"] == "tie" else row["human_better"]
    deploy_rows.append({"질문 ID": row["question_id"], "판사 선택": choice, "사람의 상대 선호": human_better,
                        "v1 사람 판정": row["label_v1"], "v2 사람 판정": row["label_v2"],
                        "사람 선호 일치": choice == human_better,
                        "판사 근거(두 순서)": " / ".join(verdict.reason for verdict in pair_verdicts_for_row)})

In [ ]:
deploy_table = pd.DataFrame(deploy_rows)
display(deploy_table)

# 7-2와 같은 함수로 집계합니다.
print_choice_counts(deploy_rows)

**확인할 점:** 7-2의 모호한 기준 결과와 비교해 `판사 근거(두 순서)`가 정책 문장과 `required_points`를 짚는지 봅니다. S07처럼 두 답이 모두 fail인 사례에서 상대 승자가 나올 수 있습니다. 상대 승자는 절대 pass를 뜻하지 않으므로, 배포 여부는 Direct Scoring 결과와 사람 검토를 함께 봐야 합니다.

### 7-4. 사람 판정으로 Direct Scoring 판사 확인하기

Pairwise만으로는 각 답이 pass인지 알 수 없습니다. 각 로그의 v1·v2 답을 한 줄씩 펼쳐 16건을 Direct Scoring으로 평가하고, 사람 판정과 비교합니다. `batch()`로 독립된 판사 요청을 묶어 실행합니다.

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다.
policy_direct_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 수업용 가상 굿즈 스토어 정책 답변을 검토합니다. 아래 평가 기준을 하나씩 대조하고, 하나라도 어기면 fail입니다. "
        "말투나 길이만으로 감점하지 말고, 판단 근거를 먼저 쓰세요.\n{criterion}"
    )),
    ("human", "제공 정책:\n{context}\n\n질문: {question}\n답변: {answer}"),
])
policy_direct_chain = ...  # DirectVerdict 구조화 출력을 연결합니다.

In [ ]:
# v1·v2 답 16건의 판정 입력을 구성합니다.
policy_check_inputs = []
for row in store_answer_rows:
    policy_check_inputs.append({
        "context": policy_text(row["policy_ids"]),
        "criterion": store_criterion(log_by_id[row["질문 ID"]]),
        "question": row["질문"],
        "answer": row["답변"],
    })
policy_check_verdicts = policy_direct_chain.batch(policy_check_inputs, config={"max_concurrency": 4})

policy_check_rows = []
for row, verdict in zip(store_answer_rows, policy_check_verdicts):
    policy_check_rows.append({
        "질문 ID": row["질문 ID"], "버전": row["버전"], "사람 판정": row["사람 판정"],
        "판사 판정": verdict.verdict, "사람 검토 이유": row["사람 검토 이유"], "판사 근거": verdict.reason,
    })
policy_check_table = pd.DataFrame(policy_check_rows)

In [ ]:
display(policy_check_table)

store_judge_report = summarize_judge_results(policy_check_rows)
display(pd.DataFrame([store_judge_report], index=["정책 기준 Direct Scoring 판사"]))

# 사람이 fail로 봤는데 판사가 pass한 답과, 그 반대 사례를 따로 모읍니다.
missed_rows = []
false_alarm_rows = []
for row in policy_check_rows:
    if row["사람 판정"] == "fail" and row["판사 판정"] == "pass":
        missed_rows.append(row)
    elif row["사람 판정"] == "pass" and row["판사 판정"] == "fail":
        false_alarm_rows.append(row)
print("놓친 실패:", len(missed_rows))
display(pd.DataFrame(missed_rows))
print("헛경보:", len(false_alarm_rows))
display(pd.DataFrame(false_alarm_rows))

**확인할 점:** 사람이 fail로 본 답을 판사가 pass로 넘긴 사례는 **놓친 실패**이고, 사람이 pass로 본 답을 fail로 본 사례는 **헛경보**입니다. 오판한 답은 판사 근거와 사람 검토 이유를 나란히 읽고, 정책 문장·`required_points`·판사 프롬프트 중 무엇을 고칠지 정합니다. 8개 질문으로 만든 수업용 자료이므로 이 결과만으로 실제 스토어의 출시 여부나 판사 기준을 확정하지 않습니다.

### 7-5. 배포 판단: Pairwise 승자와 절대 기준을 함께 읽기

7-3의 Pairwise는 "어느 버전이 더 나은가", 7-4의 Direct Scoring은 "각 답이 기준을 통과하는가"에 답합니다. 배포 여부는 두 번째 질문에 달려 있습니다. 버전별로 Pairwise 승리 수, 사람 판정과 판사 판정의 pass 수, fail 질문을 나란히 셉니다.

In [ ]:
# 버전별로 Pairwise 승리 수와 절대 판정(pass 수·fail 질문)을 한 표로 모읍니다.
release_rows = []
for version in ["v1", "v2"]:
    version_rows = [row for row in policy_check_rows if row["버전"] == version]
    release_rows.append({
        "버전": version,
        "Pairwise 승리(7-3)": sum(row["판사 선택"] == version for row in deploy_rows),
        "사람 pass": sum(row["사람 판정"] == "pass" for row in version_rows),
        "판사 pass": sum(row["판사 판정"] == "pass" for row in version_rows),
        "사람 fail 질문": ", ".join(row["질문 ID"] for row in version_rows if row["사람 판정"] == "fail"),
        "판사 fail 질문": ", ".join(row["질문 ID"] for row in version_rows if row["판사 판정"] == "fail"),
    })
display(pd.DataFrame(release_rows))

**확인할 점:** 사람 판정 기준으로 v1은 8개 질문 중 5개, v2는 3개를 통과하고, S07은 두 버전 모두 fail입니다. Pairwise에서 더 많이 이긴 버전이 있더라도 fail 질문이 남아 있으면 그대로 배포하지 않습니다. 실무에서는 fail 질문의 답을 고친 뒤 같은 사례로 다시 평가하고, 판사 판정이 사람 판정과 다른 질문은 기준이나 사례부터 점검합니다.

## 8. 핵심 정리

- **LLM-as-Judge:** 사람이 정한 기준으로 판사 모델이 LLM 응답을 채점하는 일반 평가 방법입니다. RAG 답변에도 적용할 수 있으며, 오늘은 검색기를 실행하지 않았습니다.
- **판사 프롬프트:** 역할·평가 축 하나의 기준·통과 기준·근거 자료·결과 형식을 적고, 판정 이유를 먼저 쓰게 합니다.
- **Direct Scoring과 Pairwise:** 답 하나의 기준 충족 여부와 두 후보의 상대 선호는 서로 다른 질문입니다. Pairwise 승자는 pass가 아니며, 배포 판단은 절대 기준으로 합니다.
- **점수별 기준과 기준 답변:** 점수를 쓰면 점수마다 기준 문장을 정하고, 사람이 검토한 기준 답변을 함께 줄 수 있습니다. 사람 판정과는 같은 평가 축끼리 비교합니다.
- **편향 확인:** 순서를 바꿔 두 번 묻고, 길이나 근거에 없는 출처 표현만 다른 사례를 비교합니다. 두 순서의 승자가 다르면 무승부로 처리합니다. 적은 반복만으로 편향 원인을 확정하지 않습니다.
- **사람 판정과 대조:** 일치율만 보지 않고 실패 탐지율·정상 통과율·kappa와 놓친 실패·헛경보 사례를 함께 봅니다.
- **다음 교안:** 같은 평가를 DeepEval의 `LLMTestCase`와 지표로 옮기고, `evaluate()`로 여러 사례를 묶어 실행합니다.